In [1]:
#Set up

import os
import pandas as pd
import numpy as np

base_path    = "/Users/akshayraghunath/Documents/"
data_path    = base_path + "data/raw/"
clean_path   = base_path + "data/clean/"
exports_path = base_path + "outputs/exports/"

for path in [data_path, clean_path, exports_path]:
    os.makedirs(path, exist_ok=True)
print(f" Set up Completed");

 Set up Completed


In [2]:
#Data Load

df = pd.read_csv("/Users/akshayraghunath/Documents/IBMHR_messy.csv")

print(f"Shape: {df.shape}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Shape: (1544, 37)
Rows: 1,544
Columns: 37


In [3]:
# These columns have zero analytical value

cols_to_drop = [
    'EmployeeCount',  # every row = 1
    'Over18',         # every row = Y
    'StandardHours',  # every row = 80
    'RandomNoise',    # junk column
    'EmployeeNumber'  # just an ID
]

df = df.drop(columns=cols_to_drop)

print(f"Columns remaining: {df.shape[1]}")
print(f"Dropped: {cols_to_drop}")


Columns remaining: 32
Dropped: ['EmployeeCount', 'Over18', 'StandardHours', 'RandomNoise', 'EmployeeNumber']


In [4]:
# Check before fix

print("Before fix:")
print(df['Attrition'].value_counts())

# Fix 'Yess' → 'Yes'
df['Attrition'] = df['Attrition'].replace('Yess', 'Yes')

# Verify after fix
print("\nAfter fix:")
print(df['Attrition'].value_counts())

Before fix:
No      1226
Yes      241
Yess      77
Name: Attrition, dtype: int64

After fix:
No     1226
Yes     318
Name: Attrition, dtype: int64


In [5]:
# Check missing values

missing = df.isnull().sum()
missing = missing[missing > 0]
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_%': missing_pct
})

print("Columns with missing values:")
print(missing_df)

Columns with missing values:
                missing_count  missing_%
Age                       155      10.04
Department                154       9.97
JobRole                   155      10.04
MonthlyIncome             156      10.10
YearsAtCompany            158      10.23


In [6]:
# Numeric columns → fill with median

df['Age']            = df['Age'].fillna(df['Age'].median())
df['MonthlyIncome']  = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['YearsAtCompany'] = df['YearsAtCompany'].fillna(df['YearsAtCompany'].median())

# Categorical columns → fill with mode

df['Department'] = df['Department'].fillna(df['Department'].mode()[0])
df['JobRole']    = df['JobRole'].fillna(df['JobRole'].mode()[0])

# Verify

print("Missing values remaining:")
remaining = df.isnull().sum()[df.isnull().sum() > 0]
print(remaining if len(remaining) > 0 else "None — all clean!")

Missing values remaining:
None — all clean!


In [7]:
# See what unique problem values exist
print("Unique non-date values in HireDate:")
print(df['HireDate'].unique()[:20])

# Count problem values
problem_values = df[~df['HireDate'].str.match(r'\d{4}-\d{2}-\d{2}', na=False)]
print(f"\nProblem rows: {len(problem_values)}")
print(problem_values['HireDate'].value_counts())

Unique non-date values in HireDate:
['2014-12-14 00:00:00' '2013-03-18 00:00:00' '2009-12-22 00:00:00'
 '2014-08-10 00:00:00' '2016-01-12 00:00:00' '2013-02-13 00:00:00'
 '2012-09-22 00:00:00' '2016-10-27 00:00:00' 'not_available'
 '2008-11-10 00:00:00' '2014-01-27 00:00:00' '2006-05-24 00:00:00'
 '2011-06-11 00:00:00' '2009-05-29 00:00:00' '2011-01-07 00:00:00'
 '2006-12-20 00:00:00' '2005-09-05 00:00:00' '2012-02-15 00:00:00'
 '2007-07-18 00:00:00' '2006-06-16 00:00:00']

Problem rows: 154
not_available    154
Name: HireDate, dtype: int64


In [8]:
# Step 1 → Replace 'not_available' with NaT

df['HireDate'] = df['HireDate'].replace('not_available', pd.NaT)

# Step 2 → Convert to datetime

df['HireDate'] = pd.to_datetime(df['HireDate'])

# Step 3 → Fill NaT with median hire date

median_hire_date = df['HireDate'].dropna().median()
df['HireDate'] = df['HireDate'].fillna(median_hire_date)

# Verify

print(f"Median HireDate used for fill: {median_hire_date.date()}")
print(f"Remaining nulls in HireDate: {df['HireDate'].isnull().sum()}")
print(f"HireDate dtype: {df['HireDate'].dtype}")
print(f"\nSample dates:")
print(df['HireDate'].head(5))

Median HireDate used for fill: 2011-09-18
Remaining nulls in HireDate: 0
HireDate dtype: datetime64[ns]

Sample dates:
0   2014-12-14
1   2013-03-18
2   2009-12-22
3   2014-08-10
4   2016-01-12
Name: HireDate, dtype: datetime64[ns]


In [9]:
# Convert floats → integers

df['Age']            = df['Age'].astype(int)
df['MonthlyIncome']  = df['MonthlyIncome'].astype(int)
df['YearsAtCompany'] = df['YearsAtCompany'].astype(int)

# Verify all dtypes look correct

print("Final data types:")
print(df.dtypes)
print(f"\nShape: {df.shape}")

Final data types:
Age                                  int64
Attrition                           object
BusinessTravel                      object
DailyRate                            int64
Department                          object
DistanceFromHome                     int64
Education                            int64
EducationField                      object
EnvironmentSatisfaction              int64
Gender                              object
HourlyRate                           int64
JobInvolvement                       int64
JobLevel                             int64
JobRole                             object
JobSatisfaction                      int64
MaritalStatus                       object
MonthlyIncome                        int64
MonthlyRate                          int64
NumCompaniesWorked                   int64
OverTime                            object
PercentSalaryHike                    int64
PerformanceRating                    int64
RelationshipSatisfaction            

In [10]:
print("=" * 50)
print("FINAL DATA QUALITY REPORT")
print("=" * 50)

print(f"\n Shape: {df.shape}")
print(f"   Rows:    {len(df):,}")
print(f"   Columns: {df.shape[1]}")

print(f"\n Attrition Distribution:")
attrition_counts = df['Attrition'].value_counts()
attrition_pct    = df['Attrition'].value_counts(normalize=True) * 100
for val in attrition_counts.index:
    print(f"   {val}: {attrition_counts[val]:,} ({attrition_pct[val]:.1f}%)")

print(f"\n Missing Values: {df.isnull().sum().sum()}")

print(f"\n HireDate Range:")
print(f"   Earliest: {df['HireDate'].min().date()}")
print(f"   Latest:   {df['HireDate'].max().date()}")

print(f"\n MonthlyIncome Range:")
print(f"   Min: ${df['MonthlyIncome'].min():,}")
print(f"   Max: ${df['MonthlyIncome'].max():,}")
print(f"   Avg: ${df['MonthlyIncome'].mean():,.0f}")

print(f"\n Age Range:")
print(f"   Min: {df['Age'].min()}")
print(f"   Max: {df['Age'].max()}")
print(f"   Avg: {df['Age'].mean():.1f}")

print("\n" + "=" * 50)
print(" DATA CLEAN — READY FOR ANALYSIS")
print("=" * 50)

FINAL DATA QUALITY REPORT

 Shape: (1544, 32)
   Rows:    1,544
   Columns: 32

 Attrition Distribution:
   No: 1,226 (79.4%)
   Yes: 318 (20.6%)

 Missing Values: 0

 HireDate Range:
   Earliest: 2005-01-03
   Latest:   2018-09-06

 MonthlyIncome Range:
   Min: $1,009
   Max: $19,999
   Avg: $6,404

 Age Range:
   Min: 18
   Max: 60
   Avg: 36.9

 DATA CLEAN — READY FOR ANALYSIS


In [11]:
# Export clean CSV

base_path  = "/Users/akshayraghunath/Documents/ibmhr/"
clean_path = base_path + "data/clean/"
os.makedirs(clean_path, exist_ok=True)
print(f" Folder created: {clean_path}")

clean_file = clean_path + "IBMHR_clean.csv"
df.to_csv(clean_file, index=False)
print(f" Clean CSV exported!")
print(f" Location: {clean_file}")
print(f"   Rows:    {len(df):,}")
print(f"   Columns: {df.shape[1]}")

 Folder created: /Users/akshayraghunath/Documents/ibmhr/data/clean/
 Clean CSV exported!
 Location: /Users/akshayraghunath/Documents/ibmhr/data/clean/IBMHR_clean.csv
   Rows:    1,544
   Columns: 32


In [14]:
from sqlalchemy import create_engine

engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={
        "host":     "localhost",
        "port":     5432,
        "dbname":   "IBM",
        "user":     "ibm_user",
        "password": "Aquadevida@1"
    }
)

# Test connection
with engine.connect() as conn:
    print("Connected to IBM database!")

Connected to IBM database!


In [15]:
df.to_sql('employees', engine, if_exists='replace', index=False)
print(f" Loaded to IBM database!")
print(f"   Table: employees")
print(f"   Rows:  {len(df):,}")

 Loaded to IBM database!
   Table: employees
   Rows:  1,544
